# 6. Гибридизация и выводы

В прошлом ноутбуке лучшей оказалась Item-based CF. Но у неё, как и у любой коллаборативной модели, есть случаи, где она не работает:
- новая книга без оценок. Её вектор оценок пуст, сходство со всеми книгами нулевое, и в рекомендации она не попадёт никогда.
- новый пользователь без оценок. Сходства считать не от чего.

Здесь делаем простой гибрид, который закрывает оба случая, проверяем его на искусственном холодном старте
и подводим итоги всей работы.

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.preprocessing import normalize

from utils import (
    ARTIFACTS_DIR, N_BOOKS, N_USERS, RELEVANCE_THRESHOLD, SEED, TOP_K,
    get_eval_users, keep_top_k, load_books, load_ratings, recommend_top_k, split_ratings, to_sparse,
)

ratings = load_ratings()
books = load_books()
train, test = split_ratings(ratings)

tfidf = sp.load_npz(ARTIFACTS_DIR / "tfidf.npz")       # TF-IDF профили книг из ноутбука 2
popular = pd.read_csv(ARTIFACTS_DIR / "popular.csv")   # книги по среднему рейтингу из ноутбука 2

## Стратегия

Гибрид с переключением: для каждой ситуации используется та модель, которая в ней работает.

| Ситуация | Что используем | Почему |
|---|---|---|
| Пользователь с историей, книга с оценками | Item-based CF | лучшая модель по метрикам |
| Книга без оценок или с малым их числом | контентная близость к книгам из истории пользователя | теги и название есть у книги сразу |
| Пользователь без истории | популярные книги | о пользователе ничего не известно |

В задании в качестве примера предлагается SVD для известных книг. Мы берём Item-based CF, потому что на наших данных
SVD ранжирует на уровне случайной выдачи (ноутбук 5). SVD остаётся полезным там, где нужна сама оценка,
например чтобы показать пользователю ожидаемый рейтинг книги.

Оценки двух моделей лежат в разных шкалах, поэтому для каждого пользователя обе шкалируются так,
чтобы максимум был равен 1. Для тёплых книг берётся оценка CF, для холодных контентная, умноженная на коэффициент
$\beta < 1$. Коэффициент задаёт, насколько охотно новые книги пускаются в выдачу. По одной только контентной близости
мы уверены в рекомендации меньше, чем по реальному поведению читателей.

In [2]:
K_NEIGHBORS = 20


def scale_rows(scores: np.ndarray) -> np.ndarray:
    """Делит каждую строку на её максимум, чтобы оценки разных моделей были в одной шкале."""
    return scores / np.maximum(scores.max(axis=1, keepdims=True), 1e-9)


class HybridRecommender:
    """Item-based CF для книг с оценками, контентная модель для книг без оценок, популярное для новых пользователей."""

    def __init__(self, train, tfidf, popular_ids, min_ratings=20, beta=0.3):
        self.R = to_sparse(train)
        self.tfidf, self.tfidf_t = tfidf, tfidf.T.tocsr()
        self.popular_ids = np.asarray(popular_ids)
        self.beta = beta

        # книга считается холодной, если в обучении у неё меньше min_ratings оценок
        n_ratings = np.bincount(train["book_id"] - 1, minlength=N_BOOKS)
        self.is_cold = n_ratings < min_ratings

        # сходства книг, как в ноутбуке 3: косинус по столбцам и top-K соседей
        X = normalize(self.R.tocsc(), norm="l2", axis=0)
        sim = (X.T @ X).tocsr()
        sim.setdiag(0)
        self.sim_top_t = keep_top_k(sim, K_NEIGHBORS).T.tocsr()

    def cf_scores(self, user_idx):
        return (self.R[user_idx] @ self.sim_top_t).toarray()

    def content_scores(self, user_idx):
        return ((self.R[user_idx] @ self.tfidf) @ self.tfidf_t).toarray()

    def hybrid_scores(self, user_idx):
        scores = scale_rows(self.cf_scores(user_idx))
        content = scale_rows(self.content_scores(user_idx))
        scores[:, self.is_cold] = self.beta * content[:, self.is_cold]
        return scores

    def recommend(self, user_id: int, N: int = 5) -> np.ndarray:
        """book_id рекомендованных книг. Для пользователя без истории возвращает популярные."""
        if user_id > N_USERS or self.R[user_id - 1].nnz == 0:
            return self.popular_ids[:N]
        return recommend_top_k(self.hybrid_scores, seen=self.R, user_ids=[user_id], k=N)[0]

## Проверка на холодном старте книг

В нашей выборке почти у всех книг есть 70-90 оценок, холодных книг нет. Поэтому смоделируем их:
выберем случайные 500 книг и удалим все их оценки из обучения, как будто эти книги только что добавили в каталог.
В тесте их оценки остаются. Посмотрим, сможет ли модель порекомендовать эти книги тем, кому они на самом деле понравились.

In [3]:
rng = np.random.default_rng(SEED)
new_books = rng.choice(np.arange(1, N_BOOKS + 1), size=500, replace=False)
train_without_new = train[~train["book_id"].isin(new_books)]

relevant = test[test["rating"] >= RELEVANCE_THRESHOLD]
relevant_matrix = to_sparse(relevant, values=np.ones(len(relevant)))

# оцениваем пользователей, у которых осталась история в train и есть релевантные книги в тесте
eval_users = get_eval_users(test)
eval_users = eval_users[np.isin(eval_users, train_without_new["user_id"].unique())]

is_new = np.zeros(N_BOOKS, dtype=bool)
is_new[new_books - 1] = True
n_relevant_total = relevant_matrix[eval_users - 1].sum()
n_relevant_new = relevant_matrix[eval_users - 1][:, is_new].sum()

print(f"Оценок в train: было {len(train)}, осталось {len(train_without_new)}")
print(f"Пользователей для оценки: {len(eval_users)}")
print(f"Релевантных книг в тесте: {n_relevant_total:.0f}, из них новых: {n_relevant_new:.0f} ({n_relevant_new / n_relevant_total:.1%})")

Оценок в train: было 784517, осталось 745420
Пользователей для оценки: 37959
Релевантных книг в тесте: 129543, из них новых: 6623 (5.1%)


In [4]:
def summarize(recs) -> dict:
    hits = relevant_matrix[eval_users[:, None] - 1, recs - 1].toarray().astype(bool)
    new_in_recs = is_new[recs - 1]
    return {
        "Precision@10": hits.mean(),
        "Найдено релевантных, %": hits.sum() / n_relevant_total * 100,
        "Доля новых книг в выдаче, %": new_in_recs.mean() * 100,
        "Найдено релевантных новых книг": int((hits & new_in_recs).sum()),
    }


model = HybridRecommender(train_without_new, tfidf, popular["book_id"])
print(f"Холодных книг (меньше 20 оценок в train): {model.is_cold.sum()}")

variants = {"Item-based CF": model.cf_scores, "Content-based": model.content_scores}
results = {
    name: summarize(recommend_top_k(score_fn, seen=model.R, user_ids=eval_users, k=TOP_K))
    for name, score_fn in variants.items()
}
for beta in (0.3, 0.5, 0.7):
    model.beta = beta
    recs = recommend_top_k(model.hybrid_scores, seen=model.R, user_ids=eval_users, k=TOP_K)
    results[f"Гибрид, beta = {beta}"] = summarize(recs)

cold_start_table = pd.DataFrame(results).T
cold_start_table["Найдено релевантных новых книг"] = cold_start_table["Найдено релевантных новых книг"].astype(int)
cold_start_table.round({"Precision@10": 4, "Найдено релевантных, %": 2, "Доля новых книг в выдаче, %": 2})

Холодных книг (меньше 20 оценок в train): 502


,Precision@10,"Найдено релевантных, %","Доля новых книг в выдаче, %",Найдено релевантных новых книг
Item-based CF,0.0787,23.06,0.00,0
Content-based,0.0463,13.57,6.41,953
"Гибрид, beta = 0.3",0.0787,23.07,3.07,258
"Гибрид, beta = 0.5",0.0769,22.54,10.21,676
"Гибрид, beta = 0.7",0.0721,21.12,22.24,1353


По таблице видно следующее.

- Item-based CF не нашёл ни одной из 6 623 релевантных новых книг. Без оценок в обучении сходство такой книги
  со всеми остальными равно нулю, и score у неё всегда нулевой.
- Контентная модель новые книги находит, но в целом заметно слабее.
- Гибрид с $\beta = 0.3$ сохраняет общую точность Item-based CF и при этом находит 258 релевантных новых книг,
  отдавая новинкам около 3% выдачи. При $\beta = 0.5$ новинок в выдаче уже 10% и найдено 676, но общая точность
  немного падает, а при $\beta = 0.7$ падает заметно.

Коэффициент $\beta$ задаёт компромисс между точностью сейчас и показами для новых книг. Для итоговой модели берём 0.3:
точность не теряется, а новые книги начинают попадать в выдачу. В реальном сервисе его стоило бы подбирать
на отдельной валидационной выборке и, возможно, брать больше, потому что без показов новая книга не соберёт оценок
и так и останется холодной.

## Итоговая модель и примеры

Обучаем гибрид на всём train с $\beta = 0.3$. В полном train холодных книг почти нет, поэтому для обычных пользователей
он ведёт себя как Item-based CF, а запасные ветки срабатывают только там, где нужны.

In [5]:
hybrid = HybridRecommender(train, tfidf, popular["book_id"], beta=0.3)
print(f"Холодных книг в полном train: {hybrid.is_cold.sum()}")

titles = books.set_index("book_id")["title"]
history_len = train.groupby("user_id").size()


def show_recommendations(user_id: int, N: int = 5) -> None:
    history = train[train["user_id"] == user_id].sort_values("rating", ascending=False)
    print(f"Пользователь {user_id}, оценок в истории: {len(history)}")
    for row in history.head(5).itertuples():
        print(f"    {row.rating}  {titles[row.book_id]}")
    print("  Рекомендации:")
    for book_id in hybrid.recommend(user_id, N):
        print(f"    {titles[book_id]}")
    print()


show_recommendations(int(history_len[history_len == 30].index[0]))   # активный пользователь
show_recommendations(int(history_len[history_len == 2].index[0]))    # короткая история
show_recommendations(N_USERS + 1)                                    # новый пользователь

Холодных книг в полном train: 2
Пользователь 46, оценок в истории: 30
    5  The Mystery of the Blue Train (Hercule Poirot, #6)
    5  Death in the Clouds (Hercule Poirot, #12)
    5  One, Two, Buckle My Shoe (Hercule Poirot, #22)
    5  A Caribbean Mystery (Miss Marple, #10)
    5  At Bertram's Hotel (Miss Marple, #11)
  Рекомендации:
    The Murder on the Links (Hercule Poirot, #2)
    A Pocket Full of Rye (Miss Marple, #7)
    The Moving Finger (Miss Marple, #4)
    The Clocks (Hercule Poirot, #34)
    The Big Four (Hercule Poirot, #5)

Пользователь 1, оценок в истории: 2
    4  Born on a Blue Day: Inside the Extraordinary Mind of an Autistic Savant
    3  Brunelleschi's Dome: How a Renaissance Genius Reinvented Architecture
  Рекомендации:
    Michelangelo and the Pope's Ceiling
    My Stroke of Insight: A Brain Scientist's Personal Journey
    The Judgment of Paris: The Revolutionary Decade That Gave the World Impressionism
    The Lives of the Artists
    Darkness Visible: A Memo

В примерах видны все три ветки. Активному читателю Агаты Кристи предлагаются другие её детективы, пользователю
с двумя оценками (научно-популярная книга об аутизме и книга об архитектуре Возрождения) предлагаются книги на те же темы,
а новому пользователю общий список по среднему рейтингу.

## Выводы по работе

### Какая модель лучше

| Модель | Precision@10 | Recall@10 | nDCG@10 | RMSE на тесте |
|---|---|---|---|---|
| Popularity | 0.0005 | 0.0018 | 0.0012 | - |
| Content-based (TF-IDF) | 0.0473 | 0.1688 | 0.1315 | - |
| Item-based CF | **0.0820** | **0.2728** | **0.2272** | 0.852 |
| SVD | 0.0005 | 0.0015 | 0.0010 | **0.829** |

Ответ зависит от задачи.

Для рекомендаций (top-N) лучше всего подходит Item-based CF. Она опирается на то, какие книги люди оценивают вместе,
а в книжных данных этот сигнал очень сильный (серии, один автор, одна полка). К тому же модель ранжирует книги
по близости к истории пользователя, то есть как раз пытается угадать, что он выберет дальше.

Оценку точнее всех предсказывает SVD. Но ранжирование по предсказанной оценке на наших данных не работает,
в топ попадают одни и те же высоко оценённые книги, не связанные с интересами пользователя. Модель знает,
какую оценку человек поставит книге, если прочитает её, но не знает, захочет ли он её читать. Поэтому
низкий RMSE ещё не означает хороших рекомендаций, и метрику надо выбирать под задачу.

### Сильные и слабые стороны моделей

| Модель | Сильные стороны | Слабые стороны |
|---|---|---|
| Popularity | Простая, работает для новых пользователей, не требует истории | Не персонализирована, на этой выборке не лучше случайной выдачи, средний рейтинг выносит наверх нишевые книги |
| Content-based | Работает для новых книг, рекомендации легко объяснить (тот же жанр, та же серия), не нужны чужие оценки | Зависит от качества тегов, не различает хорошие и плохие книги, мало разнообразия |
| Item-based CF | Лучшее качество top-N, сходства можно считать офлайн, результат легко объяснить (похоже на то, что вы читали) | Не работает для новых книг и пользователей, матрица сходств растёт квадратично от числа книг, шумные сходства при малом числе общих оценок |
| SVD | Лучший RMSE, компактная модель, быстро учится, хорошо масштабируется | Плохое ранжирование по всему каталогу, холодный старт, факторы трудно интерпретировать |
| Гибрид | Закрывает холодный старт книг и пользователей, не теряя качества на остальных | Больше параметров ($\beta$, порог холодности), их нужно подбирать, сложнее поддерживать |

### Как улучшить систему

1. Данные. Взять полную версию датасета (6 млн оценок, реальное распределение популярности) и данные с временными
   метками, чтобы делить выборку по времени. Использовать неявные сигналы: в датасете есть `to_read.csv` со списками
   книг, которые пользователь хочет прочитать, это прямое указание на интерес.
2. Учить модель ранжировать. Заменить SVD на разложение под неявный отклик (ALS, BPR), где оптимизируется
   порядок книг, а не ошибка оценки. Скорее всего, это даст самый большой прирост качества.
3. Доработать Item-based CF. Добавить штраф за малое число общих оценок (shrinkage), центрирование оценок по пользователю
   (adjusted cosine), подбирать число соседей по метрикам ранжирования, а не по RMSE.
4. Текстовые эмбеддинги. Вместо TF-IDF по тегам использовать эмбеддинги описаний книг (Word2Vec, BERT,
   sentence-transformers). Они учитывают синонимы (сейчас `sci-fi` и `science-fiction` считаются разными токенами) и смысл текста.
   Описаний в датасете нет, но их можно получить по идентификаторам Goodreads.
5. Двухэтапная схема. Item-based CF и контентная модель отбирают несколько сотен кандидатов, а градиентный бустинг
   ранжирует их по признакам: оценка SVD, близость по CF и по контенту, признаки пользователя (активность,
   любимые жанры) и книги (год, язык, средний рейтинг).
6. Нейросетевые подходы. Item2Vec, двухбашенные модели, а при наличии времени оценок последовательные модели
   (SASRec), которые учитывают порядок чтения.
7. Оценка. Добавить метрики разнообразия и новизны и проверить систему в A/B-тесте. Офлайн-метрики занижают
   качество, потому что мы не знаем реакцию пользователя на книги, которые он не оценивал.